In [ ]:
# Check the cluster
# v2 
attached_cluster_name = spark.conf.get(
    "spark.databricks.clusterUsageTags.clusterName", ""
)
if not attached_cluster_name.endswith("uc_support") and not (
    attached_cluster_name.startswith("bfdw_")
    and "compute_uc_jobs" in attached_cluster_name
):
    raise Exception(
        "This notebook is being executed in an incorrect cluster. Please attach it to the *uc_support cluster or one of the bfdw_*compute_uc_jobs* clusters"
    )
else:
    print(f"Cluster is: {attached_cluster_name}")

2.Code to select catolog based on the workspace

In [ ]:
workspace_catalogs = [
    e.catalog.lower()
    for e in spark.sql(f"SHOW CATALOGS").collect()
    if e.catalog not in ["main", "samples", "system", "__databricks_internal"]
]
print(f"Catalogs in the workspace: {workspace_catalogs}")
banfield_catalogs = ["banfield_catalogdev", "banfield_catalogtst", "banfield_catalog"]
target_catalog0 = [e for e in banfield_catalogs if e in workspace_catalogs]
if (not target_catalog0) or len(target_catalog0) != 1:
    raise Exception(
        f"Expecting any one of the active banfield catalog but received {len(target_catalog0)}; Banfield catalog names: {banfield_catalogs}"
    )
spark.conf.set("catlg.banfield_catalog", target_catalog0[0])
print(f"catlg.banfield_catalog: {target_catalog0[0]}")
 
bf_vwmvhcores = ["bf_vwmvhcoredev", "bf_vwmvhcoretst", "bf_vwmvhcore"]
target_catalog1 = [e for e in bf_vwmvhcores if e in workspace_catalogs]
if (not target_catalog1) or len(target_catalog1) != 1:
    raise Exception(
        f"Expecting any one of the active bf_vwmvhcore but received {len(target_catalog1)}; bf_vwmvhcore names: {bf_vwmvhcores}"
    )
spark.conf.set("catlg.bf_vwmvhcore", target_catalog1[0])
print(f"catlg.bf_vwmvhcore: {target_catalog1[0]}")


bf_vwedhs = ["bf_vwedhdev", "bf_vwedhtst", "bf_vwedh"]
target_catalog2 = [e for e in bf_vwedhs if e in workspace_catalogs]
if (not target_catalog1) or len(target_catalog2) != 1:
    raise Exception(
        f"Expecting any one of the active bf_vwmvhcore but received {len(target_catalog2)}; bf_vwmvhcore names: {bf_vwedhs}"
    )
spark.conf.set("catlg.bf_vwedh", target_catalog2[0])
print(f"catlg.bf_vwedh: {target_catalog2[0]}")

bf_vwvoyagers = ["bf_vwvoyagerdev", "bf_vwvoyagertst", "bf_vwvoyager"]
target_catalog3 = [e for e in bf_vwvoyagers if e in workspace_catalogs]
if (not target_catalog1) or len(target_catalog3) != 1:
    raise Exception(
        f"Expecting any one of the active bf_vwmvhcore but received {len(target_catalog3)}; bf_vwvoyager names: {bf_vwvoyagers}"
    )
spark.conf.set("catlg.bf_vwvoyager", target_catalog3[0])
print(f"catlg.bf_vwvoyager: {target_catalog3[0]}")

3. Create table Bronze Layer

In [ ]:
%sql
create or replace table ${catlg.banfield_catalog}.bfdw_bronze.dwworkday_emplegacyid (
 wd_employee_id string,
oracle_id string,
bp_legacy_id string,
src_filename string,
dw_create_dt timestamp,
dw_load_dt timestamp,
dw_job_id bigint,
fw_createdts timestamp,
fw_modifiedts timestamp,
fw_filename string,
rowhash string,
record_status string,
  constraint `dwworkday_emplegacyid_pk` primary key (`oracle_id`) rely)
using delta
tblproperties (
  'delta.checkpoint.writestatsasjson' = 'false',
  'delta.checkpoint.writestatsasstruct' = 'true',
  'delta.minreaderversion' = '1',
  'delta.minwriterversion' = '2',
  'delta.feature.allowColumnDefaults' = 'supported')

4. Create table for Silver Layer

In [ ]:
%sql
create or replace table ${catlg.banfield_catalog}.bfdw_silver.dwworkday_emplegacyid (
  wd_employee_id string,
oracle_id string,
bp_legacy_id string,
src_filename string,
dw_create_dt timestamp,
dw_load_dt timestamp,
dw_job_id bigint,
fw_createdts timestamp,
fw_modifiedts timestamp,
fw_filename string,
rowhash string,
record_status string,
  constraint `dwworkday_emplegacyid_silver_pk` primary key (`oracle_id`) rely)
using delta
tblproperties (
  'delta.checkpoint.writestatsasjson' = 'false',
  'delta.checkpoint.writestatsasstruct' = 'true',
  'delta.minreaderversion' = '1',
  'delta.minwriterversion' = '2',
  'delta.feature.allowColumnDefaults' = 'supported')

5. Create View for Gold Layer

In [ ]:
%sql
create or replace view ${catlg.banfield_catalog}.bfdw_gold.cdwworkday_emplegacyid

as

select a.*
from ${catlg.banfield_catalog}.bfdw_silver.cdwworkday_emplegacyid a where 1 =1 

6. Create View for bfdw_date_quality for silver layer

In [ ]:
%sql
create or replace view ${catlg.banfield_catalog}.bfdw_data_quality.dwworkday_emplegacyid_silver_primary_key_exceptions

as

select a.*
from ${catlg.banfield_catalog}.bfdw_silver.dwworkday_emplegacyid a where 1 !=1 


7. Create View for bfdw_data_quality for Bronze layers

In [ ]:
%sql
create or replace view ${catlg.banfield_catalog}.bfdw_data_quality.dwworkday_emplegacyid_bronze_record_status_exceptions

as
Select *  
from  ${catlg.banfield_catalog}.bfdw_bronze.dwworkday_emplegacyid
where record_status != 'valid';